# Temperature Sweeps

Query fridge temperature logs from the Bluefors API and correlate with experiment data.

In [8]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from bluefors import BlueforsClient, Lakeshore

%matplotlib inline

load_dotenv()

client = BlueforsClient(
    host=os.environ["BLUEFORS_HOST"],
    port=int(os.environ["BLUEFORS_PORT"]),
    api_key=os.environ["BLUEFORS_API_KEY"],
    verify_ssl=False,
    timeout=60,
)

## Connect to RFSoC

In [9]:
cfg_file = 'sample_50_rfboard.yml'
expt_path = 'C:\\_Data\\sample_50_new\\'

In [10]:
from qick import QickConfig
from slab_qick_calib.exp_handling.instrumentmanager import InstrumentManager
from slab_qick_calib.calib.time_tracking import time_tracking
from slab_qick_calib.helpers import config, handy

handy.config_figs()

configs_dir = os.path.join(os.getcwd(), '..', 'configs')
cfg_path = os.path.join(configs_dir, cfg_file)
auto_cfg = config.load(cfg_path)

im = InstrumentManager(ns_address=auto_cfg['aliases']['ip'], port=8888)
soc = im[auto_cfg['aliases']['soc']]
soccfg = QickConfig(soc.get_cfg())

cfg_dict = {'soc': soccfg, 'expt_path': expt_path, 'cfg_file': cfg_path, 'im': im}

In [11]:
qubit_list = [0,1,2]
tracking_hours = 1  # hours of tracking per temperature point

## Check MXC temperature

In [12]:
mxc_temp = client.get_mxc_temperature()
print(f"MXC Temperature: {mxc_temp:.6f} K ({mxc_temp * 1000:.2f} mK)")

MXC Temperature: 0.123019 K (123.02 mK)


## Configure Lakeshore for temperature sweep
Set heater range, PID mode, turn off still heater, and increase MXC thermometer excitation.

In [4]:
# Set heater range (6 = 10 mA) and mode (PID)
client.configure_lakeshore_output(
    heater_range=Lakeshore.RANGE_10MA,
    mode=Lakeshore.MODE_PID,
)


{'data': {'driver.lakeshore.settings.outputs.sample.write': {'name': 'driver.lakeshore.settings.outputs.sample.write',
   'type': 'Method',
   'content': {'parameters': [],
    'description': 'Write configurations to the device',
    'return': None}}}}

In [ ]:

# Set initial setpoint low so we don't jump to a high temp
client.set_lakeshore_setpoint(0.05);


{'data': {'driver.lakeshore.settings.outputs.sample.write': {'name': 'driver.lakeshore.settings.outputs.sample.write',
   'type': 'Method',
   'content': {'parameters': [],
    'description': 'Write configurations to the device',
    'return': None}}}}

In [ ]:

# Turn off still heater
client.set_still_heater_power(0);


{'data': {'mapper.temperature_control.heaters.still.setpoint_power': {'name': 'mapper.temperature_control.heaters.still.setpoint_power',
   'type': 'Value.Number.Float.Unit.power',
   'content': {'read_only': False,
    'maximum_age': 0,
    'lockable': True,
    'locked': False,
    'owner': 'mapper.temperature_control.heaters.still.setpoint_power',
    'latest_valid_value': {'value': '0.0',
     'outdated': False,
     'date': 1774280793600,
     'status': 'INDEPENDENT',
     'exception': ''},
    'latest_value': {'value': '0.0',
     'outdated': False,
     'date': 1774280793600,
     'status': 'INDEPENDENT',
     'exception': ''}}}}}

In [7]:

# Increase MXC thermometer excitation for better signal (63 uV)
client.set_lakeshore_input_excitation("in6", 4)


{'data': {'driver.lakeshore.settings.inputs.in6.write': {'name': 'driver.lakeshore.settings.inputs.in6.write',
   'type': 'Method',
   'content': {'parameters': [],
    'description': 'Write configurations to the device',
    'return': None}}}}

## Set PID temperature

In [ ]:
# Set a new PID setpoint (in Kelvin)
client.set_lakeshore_setpoint(0.1)

In [7]:
tt_data, tt_path, tt_stats, tt_id = time_tracking(
    qubit_list, cfg_dict,
    total_time=tracking_hours,
    bf_client=client,
    soc=soc, t1_max = 5, t2_max = 5
)

Starting run 0, for qubit 0. Time elapsed 0.00 hrs
Activated qubit 0: ADC ch 0 (fc=3939.1391, bw=500, bandpass), DAC-ro ch 0 (fc=3939.1391, bw=100, bandpass), DAC-qb ch 1 (fc=5847.1338, bw=200, bandpass), DC bias ch 0 (val=0 V)
R2:0.859	Fit par error:0.108	 Best fit:b'avgi'


C:\_Lib\python\qq\slab_qick_calib\analysis\fitting.py:1008: OptimizeWarning: Covariance of the parameters could not be estimated
  pOpt, pCov = sp.optimize.curve_fit(decayslopesin, xdata, ydata, p0=fitparams)


R2:0.170	Fit par error:0.726	 Best fit:b'avgi'
Starting run 0, for qubit 1. Time elapsed 0.00 hrs
Activated qubit 1: ADC ch 0 (fc=7146.7939, bw=100, bandpass), DAC-ro ch 0 (fc=7146.7939, bw=100, bandpass), DAC-qb ch 1 (fc=4100.224, bw=500, bandpass), DC bias ch 0 (val=0 V)
R2:0.860	Fit par error:0.107	 Best fit:b'avgi'
Attempted to init fitparam 3 to 0.02732049851190476, which is out of bounds 0.03 to inf. Instead init to 0.03
Attempted to init fitparam 3 to 0.02732049851190476, which is out of bounds 0.03 to inf. Instead init to 0.03
Attempted to init fitparam 3 to 0.02732049851190476, which is out of bounds 0.03 to inf. Instead init to 0.03
R2:0.632	Fit par error:0.190	 Best fit:b'avgi'
Starting run 0, for qubit 2. Time elapsed 0.01 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.946	Fit par error:0.061	 Best fit:b'avgi'
R2:0.910	Fit par error:0.060	 Be

C:\_Lib\python\qq\slab_qick_calib\analysis\fitting.py:955: RuntimeWarning: overflow encountered in exp
  * np.exp(-x / decay)


R2:0.931	Fit par error:0.054	 Best fit:b'avgi'
Starting run 5, for qubit 0. Time elapsed 0.03 hrs
Activated qubit 0: ADC ch 0 (fc=3939.1391, bw=500, bandpass), DAC-ro ch 0 (fc=3939.1391, bw=100, bandpass), DAC-qb ch 1 (fc=5847.1338, bw=200, bandpass), DC bias ch 0 (val=0 V)
R2:0.792	Fit par error:0.139	 Best fit:b'avgi'
R2:0.300	Fit par error:462.594	 Best fit:b'avgi'
R2:0.159	Fit par error:0.288	 Best fit:b'avgi'
Starting run 5, for qubit 1. Time elapsed 0.04 hrs
Activated qubit 1: ADC ch 0 (fc=7146.7939, bw=100, bandpass), DAC-ro ch 0 (fc=7146.7939, bw=100, bandpass), DAC-qb ch 1 (fc=4100.224, bw=500, bandpass), DC bias ch 0 (val=0 V)
R2:0.848	Fit par error:0.156	 Best fit:b'avgi'
Attempted to init fitparam 3 to 0.02732049851190476, which is out of bounds 0.03 to inf. Instead init to 0.03
Attempted to init fitparam 3 to 0.02732049851190476, which is out of bounds 0.03 to inf. Instead init to 0.03
Attempted to init fitparam 3 to 0.02732049851190476, which is out of bounds 0.03 to inf.

...

C:\_Lib\python\qq\slab_qick_calib\analysis\fitting.py:953: RuntimeWarning: overflow encountered in multiply
  yscale


R2:nan	Fit par error:nan	 Best fit:b'avgi'
R2:0.196	Fit par error:9.561	 Best fit:b'avgi'
Starting run 52, for qubit 2. Time elapsed 0.35 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.923	Fit par error:0.087	 Best fit:b'avgi'
R2:0.918	Fit par error:0.102	 Best fit:b'avgi'
Starting run 53, for qubit 0. Time elapsed 0.35 hrs
Activated qubit 0: ADC ch 0 (fc=3939.1391, bw=500, bandpass), DAC-ro ch 0 (fc=3939.1391, bw=100, bandpass), DAC-qb ch 1 (fc=5847.1338, bw=200, bandpass), DC bias ch 0 (val=0 V)
R2:0.532	Fit par error:0.291	 Best fit:b'avgi'
R2:0.340	Fit par error:0.699	 Best fit:b'avgi'
Starting run 53, for qubit 1. Time elapsed 0.36 hrs
Activated qubit 1: ADC ch 0 (fc=7146.7939, bw=100, bandpass), DAC-ro ch 0 (fc=7146.7939, bw=100, bandpass), DAC-qb ch 1 (fc=4100.224, bw=500, bandpass), DC bias ch 0 (val=0 V)
R2:0.876	Fit par error:0.102	 Best fit:b'

C:\_Lib\python\qq\slab_qick_calib\analysis\fitting.py:360: RuntimeWarning: overflow encountered in exp
  return y0 + yscale * np.exp(-x / decay)


R2:0.891	Fit par error:0.191	 Best fit:b'avgi'
R2:0.565	Fit par error:0.205	 Best fit:b'avgi'
Starting run 87, for qubit 2. Time elapsed 0.58 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.950	Fit par error:0.064	 Best fit:b'avgi'
R2:0.923	Fit par error:0.041	 Best fit:b'avgi'
Starting run 88, for qubit 0. Time elapsed 0.58 hrs
Activated qubit 0: ADC ch 0 (fc=3939.1391, bw=500, bandpass), DAC-ro ch 0 (fc=3939.1391, bw=100, bandpass), DAC-qb ch 1 (fc=5847.1338, bw=200, bandpass), DC bias ch 0 (val=0 V)
R2:0.737	Fit par error:0.180	 Best fit:b'avgi'
R2:0.159	Fit par error:0.319	 Best fit:b'avgi'
Starting run 88, for qubit 1. Time elapsed 0.59 hrs
Activated qubit 1: ADC ch 0 (fc=7146.7939, bw=100, bandpass), DAC-ro ch 0 (fc=7146.7939, bw=100, bandpass), DAC-qb ch 1 (fc=4100.224, bw=500, bandpass), DC bias ch 0 (val=0 V)
R2:0.754	Fit par error:0.162	 Best fi

KeyboardInterrupt: 

In [16]:
temps = [0.23]
tracking_hours=0.5
settle_time = 900          # Seconds to wait after setting each temp


for temp in temps:
    

    # Set temperature and wait for settling
    client.set_lakeshore_setpoint(temp)
    time.sleep(settle_time)


    tt_data, tt_path, tt_stats, tt_id = time_tracking(
        qubit_list, cfg_dict,
        total_time=tracking_hours,
        bf_client=client,
        soc=soc, t1_max = 5, t2_max = 5
    )

Starting run 0, for qubit 2. Time elapsed 0.00 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.769	Fit par error:0.137	 Best fit:b'avgi'
R2:0.673	Fit par error:0.095	 Best fit:b'avgi'
Starting run 1, for qubit 2. Time elapsed 0.00 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.808	Fit par error:0.145	 Best fit:b'avgi'
R2:0.837	Fit par error:0.151	 Best fit:b'avgi'
Starting run 2, for qubit 2. Time elapsed 0.00 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.794	Fit par error:0.129	 Best fit:b'avgi'
R2:0.000	Fit par error:inf	 Best fit:b'avgi'
R2:0.819	Fit par error:0.698	 Best fit:b'avgi'
Star

C:\_Lib\python\qq\slab_qick_calib\analysis\fitting.py:1013: OptimizeWarning: Covariance of the parameters could not be estimated
  pOpt, pCov = sp.optimize.curve_fit(


R2:0.843	Fit par error:0.074	 Best fit:b'avgi'
Starting run 71, for qubit 2. Time elapsed 0.08 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.814	Fit par error:0.125	 Best fit:b'avgi'
R2:0.621	Fit par error:0.112	 Best fit:b'avgi'
Starting run 72, for qubit 2. Time elapsed 0.08 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.788	Fit par error:0.144	 Best fit:b'avgi'
R2:0.561	Fit par error:0.130	 Best fit:b'avgi'
Starting run 73, for qubit 2. Time elapsed 0.08 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.837	Fit par error:0.123	 Best fit:b'avgi'
R2:0.806	Fit par error:0.071	 Best fit:b'avgi'

In [19]:
client.set_lakeshore_setpoint(0.05)

{'data': {'driver.lakeshore.settings.outputs.sample.write': {'name': 'driver.lakeshore.settings.outputs.sample.write',
   'type': 'Method',
   'content': {'parameters': [],
    'description': 'Write configurations to the device',
    'return': None}}}}

In [15]:
tracking_hours=0.5
qubit_list = [2]


tt_data, tt_path, tt_stats, tt_id = time_tracking(
    qubit_list, cfg_dict,
    total_time=tracking_hours,
    bf_client=client,
    soc=soc, t1_max = 5, t2_max = 5
)

Starting run 0, for qubit 2. Time elapsed 0.00 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.778	Fit par error:0.134	 Best fit:b'avgi'
R2:0.744	Fit par error:0.093	 Best fit:b'avgi'
Starting run 1, for qubit 2. Time elapsed 0.00 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.843	Fit par error:0.111	 Best fit:b'avgi'
R2:0.790	Fit par error:0.094	 Best fit:b'avgi'
Starting run 2, for qubit 2. Time elapsed 0.00 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.876	Fit par error:0.101	 Best fit:b'avgi'


C:\_Lib\python\qq\slab_qick_calib\analysis\fitting.py:1008: OptimizeWarning: Covariance of the parameters could not be estimated
  pOpt, pCov = sp.optimize.curve_fit(decayslopesin, xdata, ydata, p0=fitparams)


R2:0.872	Fit par error:0.081	 Best fit:b'avgi'
Starting run 3, for qubit 2. Time elapsed 0.00 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.871	Fit par error:0.107	 Best fit:b'avgi'
R2:0.891	Fit par error:0.078	 Best fit:b'avgi'
Starting run 4, for qubit 2. Time elapsed 0.00 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.894	Fit par error:0.093	 Best fit:b'avgi'
R2:0.857	Fit par error:0.087	 Best fit:b'avgi'
Starting run 5, for qubit 2. Time elapsed 0.01 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.941	Fit par error:0.090	 Best fit:b'avgi'
R2:0.817	Fit par error:0.097	 Best fit:b'avgi'
St

C:\_Lib\python\qq\slab_qick_calib\analysis\fitting.py:168: OptimizeWarning: Covariance of the parameters could not be estimated
  pOpt, pCov = sp.optimize.curve_fit(fitfunc, xdata, ydata, p0=fitparams)


R2:0.846	Fit par error:0.120	 Best fit:b'avgi'
R2:0.774	Fit par error:0.108	 Best fit:b'avgi'
Starting run 44, for qubit 2. Time elapsed 0.05 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.917	Fit par error:0.079	 Best fit:b'avgi'
R2:0.820	Fit par error:0.101	 Best fit:b'avgi'
Starting run 45, for qubit 2. Time elapsed 0.05 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.884	Fit par error:0.109	 Best fit:b'avgi'
R2:0.861	Fit par error:0.079	 Best fit:b'avgi'
Starting run 46, for qubit 2. Time elapsed 0.05 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.921	Fit par error:0.083	 Best fit:b'avgi'

...

R2:nan	Fit par error:nan	 Best fit:b'avgi'
R2:0.821	Fit par error:0.325	 Best fit:b'avgi'
Starting run 220, for qubit 2. Time elapsed 0.25 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.798	Fit par error:0.152	 Best fit:b'avgi'
R2:0.889	Fit par error:0.115	 Best fit:b'avgi'
Starting run 221, for qubit 2. Time elapsed 0.25 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.949	Fit par error:0.062	 Best fit:b'avgi'
R2:0.867	Fit par error:0.081	 Best fit:b'avgi'
Starting run 222, for qubit 2. Time elapsed 0.25 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.912	Fit par error:0.083	 Best fit:b'avgi'


C:\_Lib\python\qq\slab_qick_calib\analysis\fitting.py:1020: OptimizeWarning: Covariance of the parameters could not be estimated
  pOpt, pCov = sp.optimize.curve_fit(


R2:0.787	Fit par error:0.122	 Best fit:b'avgi'
Starting run 246, for qubit 2. Time elapsed 0.28 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.877	Fit par error:0.107	 Best fit:b'avgi'
R2:0.879	Fit par error:0.070	 Best fit:b'avgi'
Starting run 247, for qubit 2. Time elapsed 0.28 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.875	Fit par error:0.119	 Best fit:b'avgi'
R2:0.748	Fit par error:0.106	 Best fit:b'avgi'
Starting run 248, for qubit 2. Time elapsed 0.28 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.894	Fit par error:0.104	 Best fit:b'avgi'
R2:0.884	Fit par error:0.075	 Best fit:b'av

C:\_Lib\python\qq\slab_qick_calib\analysis\fitting.py:360: RuntimeWarning: overflow encountered in exp
  return y0 + yscale * np.exp(-x / decay)


R2:0.782	Fit par error:0.303	 Best fit:b'avgi'
R2:0.781	Fit par error:0.113	 Best fit:b'avgi'
Starting run 349, for qubit 2. Time elapsed 0.40 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.895	Fit par error:0.089	 Best fit:b'avgi'
R2:0.853	Fit par error:0.156	 Best fit:b'avgi'
Starting run 350, for qubit 2. Time elapsed 0.40 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.834	Fit par error:0.118	 Best fit:b'avgi'
R2:0.810	Fit par error:0.120	 Best fit:b'avgi'
Starting run 351, for qubit 2. Time elapsed 0.40 hrs
Activated qubit 2: ADC ch 0 (fc=7681.8329, bw=500, bandpass), DAC-ro ch 0 (fc=7681.8329, bw=100, bandpass), DAC-qb ch 1 (fc=3400, bw=100, bandpass), DC bias ch 3 (val=0 V)
R2:0.892	Fit par error:0.101	 Best fit:b'av

In [ ]:
client.set_lakeshore_input_excitation("in6", 2); # this is 6.3 uv

{'data': {'driver.lakeshore.settings.inputs.in6.write': {'name': 'driver.lakeshore.settings.inputs.in6.write',
   'type': 'Method',
   'content': {'parameters': [],
    'description': 'Write configurations to the device',
    'return': None}}}}

In [ ]:
client.set_lakeshore_mode(Lakeshore.MODE_OFF);

{'data': {'driver.lakeshore.settings.outputs.sample.write': {'name': 'driver.lakeshore.settings.outputs.sample.write',
   'type': 'Method',
   'content': {'parameters': [],
    'description': 'Write configurations to the device',
    'return': None}}}}

## Temperature sweep loop
Steps through temperatures, waits for settling, and runs your measurement at each point. Manages turbo pump if needed.

In [ ]:
finish_cold = True         # Cool down and restore settings at the end
turbo_on = False           # Set True if turbo is currently on
turbo_off_temp = 0.200     # Turn off turbo above this temp (K); None to skip
settle_time = 900          # Seconds to wait after setting each temp

temps = [0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45]

# Create a dedicated csv subdirectory for this sweep
from datetime import datetime
sweep_id = datetime.now().strftime("%Y_%m_%d_%H_%M") + "_temp_sweep"
print(f"Sweep ID: {sweep_id}")

for temp in temps:
    print(f"\n--- Setting temp to {temp*1000:.1f} mK ---")

    # Set temperature and wait for settling
    client.set_lakeshore_setpoint(temp)
    time.sleep(settle_time)

    mxc_temp = client.get_mxc_temperature()
    print(f"  Actual MXC: {mxc_temp*1000:.2f} mK")

    # Run time tracking at this temperature
    tt_data, tt_path, tt_stats, tt_id = time_tracking(
        qubit_list, cfg_dict,
        total_time=tracking_hours,
        bf_client=client,
        soc=soc,
        csv_subdir=sweep_id,
        t1_max = 5,
        t2_max = 5

    )
    print(f"  Tracking saved: {tt_path} (id={tt_id})")

# Cooldown and restore
if finish_cold:
    print("\n--- Cooling down ---")
    client.set_lakeshore_setpoint(0)
    client.set_lakeshore_pid(p=2, i=30)
    time.sleep(600)
    if turbo_off_temp is not None and not turbo_on:
        client.set_turbo_pump(True)
    client.set_lakeshore_mode(Lakeshore.MODE_OFF)
    client.set_lakeshore_input_excitation("in6", 4)

## Restore settings
Re-enable autoscan and restore thermometer excitation to default.

In [ ]:

# Restore thermometer excitation to default
client.set_lakeshore_input_excitation("in6", 3)

## Analyze temperature sweep

Load all tracking runs from the csv directory, compute average T1/T2 at each temperature, and plot.

In [ ]:
from slab_qick_calib.analysis import collections

# Point to the sweep's csv subdirectory
sweep_csv_dir = os.path.join(expt_path, 'Tracking', 'csv', sweep_id)

df = collections.load_temp_sweep(sweep_csv_dir, qubit_list=qubit_list)
df

In [ ]:
collections.plot_temp_sweep(df, params=['t1', 't2', 'tphi', 'f_ge'])
plt.show()